In [0]:
import requests

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

from pyspark.sql.functions import (
    lit,
    current_date
)


# =========================================================
# 1. API URL
# =========================================================

api_url = "https://reqres.in/api/users"


# =========================================================
# 2. Fetch data from API page by page
# =========================================================

all_data = []

page = 1

while True:

    response = requests.get(
        api_url,
        params={"page": page}
    )

    response.raise_for_status()

    response_json = response.json()

    # Extract only the data block
    data = response_json.get("data", [])

    # Stop when data is empty
    if not data:
        break

    # Add current page data to all_data
    all_data.extend(data)

    print(f"Page {page} fetched: {len(data)} records")

    page += 1


print("Total records:", len(all_data))


# =========================================================
# 3. Custom schema
# =========================================================

user_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("email", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("avatar", StringType(), True)
])


# =========================================================
# 4. Create DataFrame with custom schema
# =========================================================

user_df = spark.createDataFrame(
    all_data,
    schema=user_schema
)

display(user_df)

user_df.printSchema()


# =========================================================
# 5. Flatten DataFrame
# =========================================================

# The API data array has already been extracted above.
# Therefore user_df contains only the flattened user records.

display(user_df)


# =========================================================
# 6. Derive site_address from email
# =========================================================

user_df = user_df.withColumn(
    "site_address",
    lit("reqres.in")
)


# =========================================================
# 7. Add load_date
# =========================================================

user_df = user_df.withColumn(
    "load_date",
    current_date()
)


# =========================================================
# 8. Display final DataFrame
# =========================================================

display(user_df)

user_df.printSchema()


# =========================================================
# 9. DBFS path
# =========================================================

person_info_path = "/dbfs/site_info/person_info"


# =========================================================
# 10. Write DataFrame as Delta
# =========================================================

user_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(person_info_path)


# =========================================================
# 11. Read Delta data and verify
# =========================================================

person_df = spark.read \
    .format("delta") \
    .load(person_info_path)

display(person_df)

person_df.printSchema()